# COST — Architecture-Matched Gating Training

This notebook reuses the COST (Improved) configuration from notebook 01
(`COST_Master_AllExperiments_CLEAN`):
- feature definitions,
- chronological split,
- missingness + delta-t preprocessing,
- model support blocks,
- hyperparameters,
- Focal Loss + R-Drop + AdamW + OneCycleLR,
- snapshot-ensemble protocol.

It does **not** rerun the conventional baselines, Initial COST, the Initial
COST ablations, or the original three-run COST (Improved) loop.

Experiment:
**COST Improved controlled gating ON vs OFF**, with all other settings fixed.

Safety: training is OFF by default.

Note: code comments refer to development stages as "Revision I" (main COST
Improved configuration from notebook 01) and "Revision II" (the gating ON/OFF
experiments). Output folder names (e.g. `revision2_outputs`) are kept unchanged
so that saved artifacts remain traceable.

In [ ]:
# CELL 1 — SETUP
!pip install -q scikit-learn imbalanced-learn torchmetrics

import os, random, warnings, time, math, copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import autocast, GradScaler

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, roc_curve
)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# REV-6A — LOCATE SUBJECT_ID SOURCE IN RAW MIMIC
# INSERT AFTER: # 1. SETUP + EXISTING CACHE
# NO TRAINING / NO MICE / NO MODEL INFERENCE
# ============================================================

from pathlib import Path
import pandas as pd
import gzip
import os

MIMIC_BASE = Path('/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1')

print("Searching MIMIC files containing subject_id...\n")

candidate_files = []

# Search likely cohort/raw tables
for p in MIMIC_BASE.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()

    if not (
        name.endswith(".csv") or
        name.endswith(".csv.gz") or
        name.endswith(".parquet")
    ):
        continue

    try:
        # Read header only
        if name.endswith(".parquet"):
            df_head = pd.read_parquet(p).head(1)
        else:
            df_head = pd.read_csv(p, nrows=2)

        cols = [str(c).lower() for c in df_head.columns]

        if "subject_id" in cols:
            candidate_files.append({
                "file": str(p),
                "subject_id": "subject_id" in cols,
                "hadm_id": "hadm_id" in cols,
                "stay_id": "stay_id" in cols,
                "admittime": "admittime" in cols,
                "intime": "intime" in cols,
                "deathtime": "deathtime" in cols,
                "hospital_expire_flag": "hospital_expire_flag" in cols,
                "n_columns": len(cols)
            })

            print("✅", p)

    except Exception:
        # Ignore tables that cannot be header-read
        pass


candidate_df = pd.DataFrame(candidate_files)

print("\n" + "="*100)
print("SUBJECT-ID SOURCE CANDIDATES")
print("="*100)

if len(candidate_df) == 0:
    print("❌ No file containing subject_id was found.")
else:
    display(candidate_df)

    candidate_df.to_csv(
        OUT_DIR / "REV6_subject_id_source_candidates.csv",
        index=False
    )

    print(
        f"\n✅ Found {len(candidate_df)} candidate files."
    )

print("\n🛑 NO TRAINING PERFORMED.")

In [ ]:
# ============================================================
# REV-6B2 — FIND ORIGINAL YEAR/COHORT CONSTRUCTION CODE
# NO TRAINING
# ============================================================

from pathlib import Path

BASE = Path("/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1")

KEYWORDS = [
    "hf_cohort_processed",
    "anchor_year",
    "anchor_year_group",
    "years =",
    "admittime",
    "intime",
    "2008",
    "2015",
    "2016",
    "2017",
    "2018",
    "2019",
]

EXTENSIONS = {
    ".py", ".ipynb", ".txt", ".md"
}

hits = []

print("Searching original cohort/year construction code...\n")

for path in BASE.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in EXTENSIONS:
        continue

    # Skip huge/output areas if needed
    if "revision2_outputs" in str(path):
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    text_lower = text.lower()

    found = [
        k for k in KEYWORDS
        if k.lower() in text_lower
    ]

    if found:
        hits.append({
            "file": str(path),
            "keywords": ", ".join(found),
            "n_hits": len(found)
        })


import pandas as pd

hits_df = pd.DataFrame(hits)

print("=" * 100)
print("POSSIBLE ORIGINAL COHORT / YEAR-MAPPING CODE")
print("=" * 100)

if len(hits_df):

    hits_df = hits_df.sort_values(
        "n_hits",
        ascending=False
    )

    display(hits_df)

else:
    print("❌ No candidate code files found.")


print("\n🛑 NO TRAINING PERFORMED.")

In [ ]:
# ============================================================
# REV-6B3 — INSPECT YEAR-MAPPING CANDIDATE
# NO TRAINING
# ============================================================

from pathlib import Path

# Ambil kandidat terbaik dari hasil REV-6B2
candidate_path = Path(hits_df.iloc[0]["file"])

print("=" * 100)
print("CANDIDATE FILE")
print("=" * 100)
print(candidate_path)

text = candidate_path.read_text(
    encoding="utf-8",
    errors="ignore"
)

keywords = [
    "anchor_year",
    "anchor_year_group",
    "hf_cohort_processed",
    "X_temporal",
    "years",
    "subject_id"
]

print("\n" + "=" * 100)
print("RELEVANT CODE / TEXT")
print("=" * 100)

lines = text.splitlines()

shown = set()

for keyword in keywords:
    matches = [
        i for i, line in enumerate(lines)
        if keyword.lower() in line.lower()
    ]

    if not matches:
        continue

    print(f"\n>>> KEYWORD: {keyword}")
    print("-" * 100)

    for idx in matches[:10]:

        start = max(0, idx - 8)
        end = min(len(lines), idx + 15)

        key = (start, end)

        if key in shown:
            continue

        shown.add(key)

        for j in range(start, end):
            print(f"{j+1:6d} | {lines[j]}")

        print("-" * 100)

print("\n🛑 NO TRAINING PERFORMED.")

In [ ]:
# ============================================================
# REV-6C — FIND ORIGINAL / INTERMEDIATE COHORT WITH SUBJECT IDs
# NO TRAINING
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import os

BASE = Path("/content/drive/MyDrive/COST_PROJECT")
TARGET_N = 14707

print("=" * 100)
print("SEARCHING FOR INTERMEDIATE COHORT / SUBJECT-ID FILES")
print("=" * 100)

results = []

# ------------------------------------------------------------
# 1. Inspect NPZ files
# ------------------------------------------------------------
for p in BASE.rglob("*.npz"):
    try:
        d = np.load(p, allow_pickle=True)

        keys = list(d.files)

        shapes = {}
        lengths = []

        for k in keys:
            try:
                arr = d[k]
                shapes[k] = str(arr.shape)

                if arr.ndim > 0:
                    lengths.append(arr.shape[0])
            except:
                pass

        interesting = any(
            any(word in k.lower() for word in [
                "subject", "patient", "hadm", "stay",
                "year", "label", "target", "id"
            ])
            for k in keys
        )

        has_14707 = TARGET_N in lengths

        if interesting or has_14707:
            results.append({
                "type": "NPZ",
                "file": str(p),
                "keys": ", ".join(keys),
                "contains_14707": has_14707,
                "details": str(shapes)[:1000]
            })

    except Exception:
        pass


# ------------------------------------------------------------
# 2. Inspect CSV / CSV.GZ / Parquet headers
# ------------------------------------------------------------
patterns = ["*.csv", "*.csv.gz", "*.parquet"]

for pattern in patterns:

    for p in BASE.rglob(pattern):

        try:
            name = p.name.lower()

            if name.endswith(".parquet"):
                df = pd.read_parquet(p).head(5)
            else:
                df = pd.read_csv(p, nrows=5)

            cols = list(df.columns)
            lower_cols = [str(c).lower() for c in cols]

            interesting = any(
                any(word in c for word in [
                    "subject_id",
                    "patient_id",
                    "hadm_id",
                    "stay_id"
                ])
                for c in lower_cols
            )

            if interesting:
                results.append({
                    "type": "TABLE",
                    "file": str(p),
                    "keys": ", ".join(map(str, cols)),
                    "contains_14707": "unknown",
                    "details": f"{len(cols)} columns"
                })

        except Exception:
            pass


# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

result_df = pd.DataFrame(results)

if len(result_df) == 0:

    print("\n❌ No candidate cohort/ID files found.")

else:

    # Prioritize files whose arrays have exactly 14,707 rows
    result_df["_priority"] = (
        result_df["contains_14707"]
        .astype(str)
        .eq("True")
        .astype(int)
    )

    result_df = (
        result_df
        .sort_values("_priority", ascending=False)
        .drop(columns="_priority")
        .reset_index(drop=True)
    )

    print(f"\nFound {len(result_df)} candidate files.\n")

    pd.set_option("display.max_colwidth", 180)

    display(
        result_df[
            [
                "type",
                "file",
                "keys",
                "contains_14707",
                "details"
            ]
        ]
    )


# ------------------------------------------------------------
# 4. Show highest-priority NPZ files separately
# ------------------------------------------------------------

if len(result_df):

    exact = result_df[
        result_df["contains_14707"].astype(str) == "True"
    ]

    print("\n" + "=" * 100)
    print("HIGHEST PRIORITY — FILES CONTAINING ARRAYS OF N = 14,707")
    print("=" * 100)

    if len(exact):
        display(exact)
    else:
        print("❌ No other NPZ with an array length of exactly 14,707.")


print("\n🛑 NO TRAINING PERFORMED.")

In [ ]:
# ============================================================
# REV-6D — VERIFY ORIGINAL 48H COHORT IDENTIFIERS
# NO TRAINING / NO MICE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PARQUET = Path(
    "/content/drive/MyDrive/COST_PROJECT/"
    "mimic_hf_q3_outputs/hf_temporal_48h_hourly.parquet"
)

print("=" * 90)
print("REV-6D — VERIFY 48H SOURCE COHORT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Read ONLY identifier/time columns
# ------------------------------------------------------------

df_id = pd.read_parquet(
    PARQUET,
    columns=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "hour"
    ]
)

print("\nRows:", f"{len(df_id):,}")

print(
    "Unique subject_id:",
    f"{df_id['subject_id'].nunique():,}"
)

print(
    "Unique hadm_id:",
    f"{df_id['hadm_id'].nunique():,}"
)

print(
    "Unique stay_id:",
    f"{df_id['stay_id'].nunique():,}"
)


# ------------------------------------------------------------
# 2. Check rows/hours per stay
# ------------------------------------------------------------

stay_summary = (
    df_id.groupby(
        ["subject_id", "hadm_id", "stay_id"],
        sort=False
    )
    .agg(
        n_rows=("hour", "size"),
        min_hour=("hour", "min"),
        max_hour=("hour", "max"),
        unique_hours=("hour", "nunique")
    )
    .reset_index()
)

print("\n" + "=" * 90)
print("48-HOUR STRUCTURE")
print("=" * 90)

print(
    "Number of stay records:",
    f"{len(stay_summary):,}"
)

print("\nRows per stay:")
print(
    stay_summary["n_rows"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nUnique hours per stay:")
print(
    stay_summary["unique_hours"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ------------------------------------------------------------
# 3. Exact 48-hour records
# ------------------------------------------------------------

exact48 = stay_summary[
    (stay_summary["n_rows"] == 48) &
    (stay_summary["unique_hours"] == 48)
].copy()

print("\n" + "=" * 90)
print("EXACT 48-HOUR STAYS")
print("=" * 90)

print(
    "Exact 48-hour stays:",
    f"{len(exact48):,}"
)

print(
    "Unique subjects among exact48:",
    f"{exact48['subject_id'].nunique():,}"
)


# ------------------------------------------------------------
# 4. Compare against processed cache N
# ------------------------------------------------------------

TARGET_N = 14707

print("\n" + "=" * 90)
print("CACHE ALIGNMENT")
print("=" * 90)

print(f"Target cache N      : {TARGET_N:,}")
print(f"All source stays    : {len(stay_summary):,}")
print(f"Exact-48h stays     : {len(exact48):,}")

if len(stay_summary) == TARGET_N:
    print(
        "\n✅ EXACT MATCH: number of source stays "
        "equals processed cache N."
    )

elif len(exact48) == TARGET_N:
    print(
        "\n✅ EXACT MATCH after requiring 48 hourly rows."
    )

else:
    print(
        "\n⚠️ Counts do not yet exactly match 14,707."
    )


# ------------------------------------------------------------
# 5. Check repeated subjects
# ------------------------------------------------------------

subject_stays = (
    stay_summary
    .groupby("subject_id")
    .size()
)

repeated = subject_stays[
    subject_stays > 1
]

print("\n" + "=" * 90)
print("REPEATED SUBJECT CHECK")
print("=" * 90)

print(
    "Subjects with >1 stay:",
    f"{len(repeated):,}"
)

print(
    "Maximum stays for one subject:",
    int(subject_stays.max())
)


# ------------------------------------------------------------
# 6. Save identifiers for later audit
# ------------------------------------------------------------

stay_summary.to_csv(
    "/content/drive/MyDrive/COST_PROJECT/"
    "mimic-iv-2.1/revision2_outputs/"
    "REV6D_source_stay_identifiers.csv",
    index=False
)

print(
    "\n✅ Identifier audit saved."
)

print("🛑 NO TRAINING PERFORMED.")

In [ ]:
# ============================================================
# REV-6E — TRACE THE 17,376 → 14,707 COHORT FILTER
# NO TRAINING / NO MICE / NO MODEL
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os

BASE = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic_hf_q3_outputs"
)

TARGET_N = 14707

print("=" * 100)
print("REV-6E — TRACE FINAL COHORT FILTER")
print("=" * 100)
print("Folder:", BASE)

rows = []

# ------------------------------------------------------------
# Inspect parquet files
# ------------------------------------------------------------

for p in BASE.rglob("*.parquet"):

    try:
        df = pd.read_parquet(p)

        n_rows = len(df)

        cols = list(df.columns)

        entry = {
            "file": p.name,
            "path": str(p),
            "rows": n_rows,
            "columns": len(cols),
            "subject_n": np.nan,
            "hadm_n": np.nan,
            "stay_n": np.nan,
            "has_label": False,
            "has_year": False,
        }

        if "subject_id" in df.columns:
            entry["subject_n"] = df["subject_id"].nunique()

        if "hadm_id" in df.columns:
            entry["hadm_n"] = df["hadm_id"].nunique()

        if "stay_id" in df.columns:
            entry["stay_n"] = df["stay_id"].nunique()

        # Search likely label columns
        lower_cols = [str(c).lower() for c in cols]

        entry["has_label"] = any(
            x in lower_cols
            for x in [
                "label",
                "labels",
                "mortality",
                "hospital_expire_flag",
                "target",
                "outcome"
            ]
        )

        entry["has_year"] = any(
            "year" in x for x in lower_cols
        )

        rows.append(entry)

        del df

    except Exception as e:

        rows.append({
            "file": p.name,
            "path": str(p),
            "rows": "ERROR",
            "columns": "ERROR",
            "subject_n": np.nan,
            "hadm_n": np.nan,
            "stay_n": np.nan,
            "has_label": False,
            "has_year": False,
        })


summary = pd.DataFrame(rows)

print("\nPARQUET INVENTORY")
print("-" * 100)

display(summary)


# ------------------------------------------------------------
# Highlight anything matching 14,707
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("POSSIBLE FINAL-COHORT FILES")
print("=" * 100)

matches = summary[
    (summary["rows"] == TARGET_N) |
    (summary["subject_n"] == TARGET_N) |
    (summary["hadm_n"] == TARGET_N) |
    (summary["stay_n"] == TARGET_N)
]

if len(matches):

    print("✅ Found candidate(s) matching N=14,707")
    display(matches)

else:

    print("❌ No direct N=14,707 parquet match.")


# ------------------------------------------------------------
# Also inspect CSV files in output folder
# ------------------------------------------------------------

csv_rows = []

for pattern in ["*.csv", "*.csv.gz"]:

    for p in BASE.rglob(pattern):

        try:
            df = pd.read_csv(p)

            csv_rows.append({
                "file": p.name,
                "path": str(p),
                "rows": len(df),
                "columns": len(df.columns),
                "subject_n":
                    df["subject_id"].nunique()
                    if "subject_id" in df.columns else np.nan,
                "hadm_n":
                    df["hadm_id"].nunique()
                    if "hadm_id" in df.columns else np.nan,
                "stay_n":
                    df["stay_id"].nunique()
                    if "stay_id" in df.columns else np.nan,
                "column_names":
                    ", ".join(map(str, df.columns))
            })

            del df

        except Exception:
            pass


csv_summary = pd.DataFrame(csv_rows)

print("\n" + "=" * 100)
print("CSV INVENTORY")
print("=" * 100)

if len(csv_summary):
    display(csv_summary)
else:
    print("No CSV files found.")


# ------------------------------------------------------------
# Search code / notebook / text for 14707 or cohort logic
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CODE / LOG REFERENCES TO COHORT CONSTRUCTION")
print("=" * 100)

keywords = [
    "14707",
    "17376",
    "drop_duplicates",
    "first stay",
    "first_stay",
    "first icu",
    "first_icu",
    "hospital_expire_flag",
    "hf_temporal_48h_hourly",
    "hf_cohort_processed",
]

code_hits = []

SEARCH_ROOT = Path(
    "/content/drive/MyDrive/COST_PROJECT"
)

for p in SEARCH_ROOT.rglob("*"):

    if (
        not p.is_file()
        or p.suffix.lower()
        not in {".py", ".ipynb", ".txt", ".md", ".log"}
    ):
        continue

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        found = [
            k for k in keywords
            if k.lower() in text.lower()
        ]

        if found:
            code_hits.append({
                "file": str(p),
                "matched": ", ".join(found),
                "n_keywords": len(found)
            })

    except Exception:
        pass


code_df = pd.DataFrame(code_hits)

if len(code_df):

    code_df = code_df.sort_values(
        "n_keywords",
        ascending=False
    )

    display(code_df)

else:
    print("No relevant code/log references found.")


print("\n🛑 NO TRAINING PERFORMED.")

In [ ]:
# CELL 2 — DRIVE + EXISTING REVISION-I CACHE ONLY
from google.colab import drive
drive.mount('/content/drive')

MIMIC_BASE = Path('/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1')
CACHE_FILE = MIMIC_BASE / 'mimic_cache' / 'hf_cohort_processed.npz'
OUT_DIR = MIMIC_BASE / 'revision2_outputs'
OUT_DIR.mkdir(exist_ok=True, parents=True)

assert CACHE_FILE.exists(), f"STOP: cache not found: {CACHE_FILE}"

d = np.load(CACHE_FILE, allow_pickle=True)
X_temporal = d['X_temporal']
X_static   = d['X_static']
labels     = d['labels']
years      = d['years']

print("Cache loaded:", X_temporal.shape, X_static.shape, labels.shape)
print("Deaths:", int(labels.sum()), f"({labels.mean()*100:.2f}%)")
print("Years:", int(years.min()), "-", int(years.max()))
print("Output:", OUT_DIR)
print("RAW MIMIC REBUILD: DISABLED")


In [ ]:
# ============================================================
# REV-6H — RECOVER EXACT FINAL 14,707 IDs
# Replicate final measurement-availability filter
# NO TRAINING / NO MICE / NO MODEL
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/COST_PROJECT")

COHORT_FILE = (
    BASE / "mimic_hf_q3_outputs/hf_cohort_index.csv"
)

TEMP_FILE = (
    BASE / "mimic_hf_q3_outputs/hf_temporal_48h_hourly.parquet"
)

CACHE_FILE = (
    BASE / "mimic-iv-2.1/mimic_cache/hf_cohort_processed.npz"
)

OUT_DIR = (
    BASE / "mimic-iv-2.1/revision2_outputs"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. LOAD
# ============================================================

print("=" * 90)
print("REV-6H — RECOVER FINAL COHORT IDENTIFIERS")
print("=" * 90)

cohort = pd.read_csv(COHORT_FILE)

temp = pd.read_parquet(TEMP_FILE)

cache = np.load(CACHE_FILE, allow_pickle=True)

cache_y = np.asarray(cache["labels"]).reshape(-1)
cache_years = np.asarray(cache["years"]).reshape(-1)

print(f"\nSource cohort : {len(cohort):,}")
print(f"Temporal rows : {len(temp):,}")
print(f"Cache N       : {len(cache_y):,}")

# ============================================================
# 2. IDENTIFY TEMPORAL FEATURE COLUMNS
# ============================================================

id_cols = {
    "subject_id",
    "hadm_id",
    "stay_id",
    "hour"
}

feature_cols = [
    c for c in temp.columns
    if c not in id_cols
]

print(f"\nTemporal feature columns: {len(feature_cols)}")
print(feature_cols)

# ============================================================
# 3. CLASSIFY VITAL vs LAB COLUMNS
#
# Based on known 48h source schema.
# We first use standard vital names.
# Everything else numeric is treated as lab candidate.
# ============================================================

vital_names = {
    "heart_rate",
    "sbp",
    "dbp",
    "map",
    "resp_rate",
    "spo2",
    "temperature"
}

vital_cols = [
    c for c in feature_cols
    if c.lower() in vital_names
]

lab_cols = [
    c for c in feature_cols
    if c not in vital_cols
]

print("\nVital columns:")
print(vital_cols)

print("\nLab candidate columns:")
print(lab_cols)

# ============================================================
# 4. COUNT OBSERVED MEASUREMENTS PER STAY
#
# Important:
# Count NON-MISSING observations across 48h.
# ============================================================

temp["_vital_measurements"] = (
    temp[vital_cols]
    .notna()
    .sum(axis=1)
)

temp["_lab_measurements"] = (
    temp[lab_cols]
    .notna()
    .sum(axis=1)
)

counts = (
    temp
    .groupby(
        ["subject_id", "hadm_id", "stay_id"],
        as_index=False
    )
    .agg(
        vital_measurements=(
            "_vital_measurements", "sum"
        ),
        lab_measurements=(
            "_lab_measurements", "sum"
        )
    )
)

print("\nStay-level measurement counts:")
display(counts.head())

# ============================================================
# 5. APPLY ORIGINAL FINAL FILTER
#    >= 6 vital measurements
#    >= 2 lab measurements
# ============================================================

eligible = counts[
    (counts["vital_measurements"] >= 6) &
    (counts["lab_measurements"] >= 2)
].copy()

print("\n" + "=" * 90)
print("ELIGIBILITY FILTER")
print("=" * 90)

print(f"All stays     : {len(counts):,}")
print(f"Eligible stays: {len(eligible):,}")
print(
    f"Removed       : "
    f"{len(counts) - len(eligible):,}"
)

# ============================================================
# 6. MERGE BACK IDENTIFIERS / OUTCOME
# ============================================================

final_ids = cohort.merge(
    eligible[
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "vital_measurements",
            "lab_measurements"
        ]
    ],
    on=["subject_id", "hadm_id", "stay_id"],
    how="inner",
    validate="one_to_one"
)

print("\nRecovered final cohort:", f"{len(final_ids):,}")

# ============================================================
# 7. CRITICAL N CHECK
# ============================================================

TARGET_N = len(cache_y)

print("\n" + "=" * 90)
print("CACHE ALIGNMENT")
print("=" * 90)

print(f"Recovered IDs : {len(final_ids):,}")
print(f"Cache target  : {TARGET_N:,}")

if len(final_ids) != TARGET_N:

    print("\n❌ NOT EXACTLY 14,707.")
    print(
        "Do NOT calculate subject overlap yet."
    )

    print(
        "\nThe vital/lab column classification or "
        "measurement-count definition differs from the "
        "original pipeline."
    )

else:

    print("\n✅ EXACT N MATCH: 14,707")

    # ========================================================
    # 8. TEST WHETHER ROW ORDER ALSO MATCHES CACHE LABELS
    # ========================================================

    source_y = (
        final_ids["in_hospital_mortality"]
        .to_numpy()
        .astype(int)
    )

    label_match = np.array_equal(
        source_y,
        cache_y.astype(int)
    )

    print(
        "Exact label sequence match:",
        "✅ YES" if label_match else "❌ NO"
    )

    # ========================================================
    # 9. IF EXACT ORDER MATCHES, ATTACH CACHE YEARS
    # ========================================================

    if label_match:

        final_ids["cache_year"] = cache_years

        final_ids["final_split"] = np.select(
            [
                final_ids["cache_year"].between(
                    2008, 2015
                ),
                final_ids["cache_year"].between(
                    2016, 2017
                ),
                final_ids["cache_year"].between(
                    2018, 2019
                )
            ],
            [
                "Training",
                "Validation",
                "Test"
            ],
            default="Unknown"
        )

        # ====================================================
        # 10. SUBJECT SETS
        # ====================================================

        tr = set(
            final_ids.loc[
                final_ids["final_split"] == "Training",
                "subject_id"
            ]
        )

        va = set(
            final_ids.loc[
                final_ids["final_split"] == "Validation",
                "subject_id"
            ]
        )

        te = set(
            final_ids.loc[
                final_ids["final_split"] == "Test",
                "subject_id"
            ]
        )

        tv = tr & va
        tt = tr & te
        vt = va & te

        # ====================================================
        # 11. SPLIT CHARACTERISTICS
        # ====================================================

        rows = []

        for split_name in [
            "Training",
            "Validation",
            "Test"
        ]:

            d = final_ids[
                final_ids["final_split"] == split_name
            ]

            rows.append({
                "Split": split_name,
                "Records": len(d),
                "Unique Subjects":
                    d["subject_id"].nunique(),
                "Mortality Positive":
                    int(
                        d[
                            "in_hospital_mortality"
                        ].sum()
                    ),
                "Prevalence (%)":
                    100 *
                    d[
                        "in_hospital_mortality"
                    ].mean()
            })

        split_table = pd.DataFrame(rows)

        print("\n" + "=" * 90)
        print("REV-6 — SPLIT CHARACTERISTICS")
        print("=" * 90)

        display(split_table)

        # ====================================================
        # 12. OVERLAP AUDIT
        # ====================================================

        overlap_table = pd.DataFrame([
            {
                "Comparison":
                    "Training–Validation",
                "Subject Overlap":
                    len(tv),
                "Status":
                    "PASS" if len(tv) == 0
                    else "OVERLAP DETECTED"
            },
            {
                "Comparison":
                    "Training–Test",
                "Subject Overlap":
                    len(tt),
                "Status":
                    "PASS" if len(tt) == 0
                    else "OVERLAP DETECTED"
            },
            {
                "Comparison":
                    "Validation–Test",
                "Subject Overlap":
                    len(vt),
                "Status":
                    "PASS" if len(vt) == 0
                    else "OVERLAP DETECTED"
            }
        ])

        print("\n" + "=" * 90)
        print("REV-6 — SUBJECT-LEVEL OVERLAP AUDIT")
        print("=" * 90)

        display(overlap_table)

        # ====================================================
        # 13. SAVE
        # ====================================================

        final_ids.to_csv(
            OUT_DIR /
            "REV6_exact_final_identifiers.csv",
            index=False
        )

        split_table.to_csv(
            OUT_DIR /
            "REV6_split_characteristics.csv",
            index=False
        )

        overlap_table.to_csv(
            OUT_DIR /
            "REV6_subject_overlap_audit.csv",
            index=False
        )

        print("\n✅ REV-6 files saved.")

        if (
            len(tv) == 0 and
            len(tt) == 0 and
            len(vt) == 0
        ):
            print(
                "\n✅ FINAL: NO CROSS-SPLIT "
                "SUBJECT OVERLAP."
            )
        else:
            print(
                "\n⚠️ FINAL: CROSS-SPLIT "
                "SUBJECT OVERLAP DETECTED."
            )

    else:

        print(
            "\n⚠️ N matches but label sequence does not."
        )

        print(
            "Do NOT attach cache years by row order yet."
        )

print("\n🛑 NO TRAINING PERFORMED.")

In [ ]:
# ============================================================
# REV-6J FINAL — EXACT SUBJECT-LEVEL LEAKAGE AUDIT
# Rebuild original cohort + original minimum-data filter
# while PRESERVING subject_id / hadm_id / stay_id.
#
# EXPECTED FINAL N = 14,707
# NO TRAINING / NO MICE / NO MODEL
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc

# ============================================================
# 0. PATHS / CONSTANTS
# ============================================================

BASE = Path("/content/drive/MyDrive/COST_PROJECT")
MIMIC_BASE = BASE / "mimic-iv-2.1"
HOSP = MIMIC_BASE / "hosp"
ICU  = MIMIC_BASE / "icu"

CACHE_FILE = (
    MIMIC_BASE / "mimic_cache/hf_cohort_processed.npz"
)

OUT_DIR = (
    MIMIC_BASE / "revision2_outputs"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 48
CHUNK = 2_000_000

CHART_ITEMIDS = {
    'heart_rate' : [220045],
    'sbp'        : [220179, 220050],
    'dbp'        : [220180, 220051],
    'map'        : [220052, 220181, 225312],
    'resp_rate'  : [220210, 224690],
    'spo2'       : [220277],
    'temperature': [223761, 223762],
}

LAB_ITEMIDS = {
    'creatinine'      : [50912],
    'bnp'             : [51002],
    'lactate'         : [50813],
    'troponin'        : [51003, 52598],
    'sodium'          : [50824, 50983],
    'potassium'       : [50822, 50971],
    'chloride'        : [50806, 50902],
    'bicarbonate'     : [50803, 50882],
    'bun'             : [51006],
    'glucose'         : [50809, 50931],
    'hemoglobin'      : [51222],
    'wbc'             : [51301],
    'platelets'       : [51265],
    'inr'             : [51237],
    'pt'              : [51274],
    'ptt'             : [51275],
    'ph'              : [50820],
    'pao2'            : [50821],
    'paco2'           : [50818],
    'fio2'            : [50816],
    'albumin'         : [50862],
    'bilirubin_total' : [50885],
    'alt'             : [50861],
    'ast'             : [50878],
}

VITAL_SIGNS = list(CHART_ITEMIDS.keys())
LAB_FEATURES = list(LAB_ITEMIDS.keys())

ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_FEATURES
N_FEATURES = len(ALL_TEMPORAL_FEATURES)

print("=" * 100)
print("REV-6J FINAL — EXACT SUBJECT-LEVEL DATA-LEAKAGE AUDIT")
print("=" * 100)
print(f"Vitals : {len(VITAL_SIGNS)}")
print(f"Labs   : {len(LAB_FEATURES)}")
print(f"Total  : {N_FEATURES}")
print("\n🛑 NO TRAINING WILL BE PERFORMED.")

t_start = time.time()

# ============================================================
# 1. LOAD ORIGINAL TABLES
# ============================================================

print("\n[1/7] Loading admissions / patients / ICU / diagnoses...")

adm = pd.read_csv(
    HOSP / "admissions.csv",
    usecols=[
        "subject_id", "hadm_id",
        "admittime", "dischtime",
        "hospital_expire_flag",
        "admission_type", "deathtime"
    ],
    parse_dates=[
        "admittime", "dischtime", "deathtime"
    ]
)

pat = pd.read_csv(
    HOSP / "patients.csv",
    usecols=[
        "subject_id", "gender",
        "anchor_age", "anchor_year",
        "anchor_year_group"
    ]
)

icu_stays = pd.read_csv(
    ICU / "icustays.csv",
    usecols=[
        "subject_id", "hadm_id", "stay_id",
        "intime", "outtime", "los"
    ],
    parse_dates=["intime", "outtime"]
)

dx = pd.read_csv(
    HOSP / "diagnoses_icd.csv",
    usecols=[
        "subject_id", "hadm_id",
        "icd_code", "icd_version"
    ]
)

print("   Admissions :", f"{len(adm):,}")
print("   Patients   :", f"{len(pat):,}")
print("   ICU stays  :", f"{len(icu_stays):,}")

# ============================================================
# 2. EXACT ORIGINAL COHORT SELECTION
# ============================================================

print("\n[2/7] Reconstructing original HF cohort...")

hf_icd10 = dx[
    (dx["icd_version"] == 10) &
    dx["icd_code"].str.startswith("I50", na=False)
]

hf_icd9 = dx[
    (dx["icd_version"] == 9) &
    dx["icd_code"].str.startswith("428", na=False)
]

hf_hadm = set(
    hf_icd10["hadm_id"].tolist() +
    hf_icd9["hadm_id"].tolist()
)

cohort = icu_stays[
    icu_stays["hadm_id"].isin(hf_hadm)
].copy()

cohort = cohort.merge(
    adm[
        [
            "hadm_id",
            "admittime",
            "hospital_expire_flag",
            "admission_type",
            "deathtime"
        ]
    ],
    on="hadm_id"
)

cohort = cohort.merge(
    pat,
    on="subject_id"
)

# Original inclusion criteria
cohort["age"] = cohort["anchor_age"]

cohort = cohort[
    cohort["age"] >= 18
].copy()

cohort = cohort[
    cohort["los"] > 1.0
].copy()

# Exact original reconstructed calendar year
cohort["admit_year"] = cohort["intime"].dt.year

cohort["anchor_year_group_start"] = (
    cohort["anchor_year_group"]
    .apply(lambda x: int(str(x).split(" - ")[0]))
)

cohort["real_admit_year"] = (
    cohort["admit_year"]
    - cohort["anchor_year"]
    + cohort["anchor_year_group_start"]
)

cohort = cohort[
    cohort["real_admit_year"].between(2008, 2019)
].copy()

# EXACT original code:
# first ICU stay per admission/hadm_id
cohort = (
    cohort
    .sort_values("intime")
    .groupby("hadm_id")
    .first()
    .reset_index()
)

cohort["label"] = (
    cohort["hospital_expire_flag"]
    .astype(int)
)

print(
    "   Before minimum-data filter:",
    f"{len(cohort):,}"
)

print(
    "   Unique subjects:",
    f"{cohort['subject_id'].nunique():,}"
)

# ============================================================
# 3. EXTRACT ORIGINAL CHART EVENTS
# ============================================================

print("\n[3/7] Reading relevant chartevents...")

stay_ids = cohort["stay_id"].tolist()
stay_id_set = set(stay_ids)

all_chart_ids = [
    iid
    for ids in CHART_ITEMIDS.values()
    for iid in ids
]

chart_chunks = []

for i, chunk in enumerate(
    pd.read_csv(
        ICU / "chartevents.csv",
        usecols=[
            "stay_id",
            "charttime",
            "itemid",
            "valuenum"
        ],
        parse_dates=["charttime"],
        chunksize=CHUNK
    )
):

    chunk = chunk[
        chunk["stay_id"].isin(stay_id_set) &
        chunk["itemid"].isin(all_chart_ids)
    ].dropna(subset=["valuenum"])

    if len(chunk):
        chart_chunks.append(chunk)

    if i % 10 == 0:
        n = sum(len(x) for x in chart_chunks)
        print(
            f"   chunk {i:3d} | "
            f"relevant rows {n:,}"
        )

chart = pd.concat(
    chart_chunks,
    ignore_index=True
)

itemid2feat = {
    iid: feature
    for feature, ids in CHART_ITEMIDS.items()
    for iid in ids
}

chart["feature"] = (
    chart["itemid"].map(itemid2feat)
)

print(
    "   Final chart rows:",
    f"{len(chart):,}"
)

del chart_chunks
gc.collect()

# ============================================================
# 4. EXTRACT ORIGINAL LAB EVENTS
# ============================================================

print("\n[4/7] Reading relevant labevents...")

hadm_ids = cohort["hadm_id"].tolist()
hadm_id_set = set(hadm_ids)

all_lab_ids = [
    iid
    for ids in LAB_ITEMIDS.values()
    for iid in ids
]

lab_chunks = []

for i, chunk in enumerate(
    pd.read_csv(
        HOSP / "labevents.csv",
        usecols=[
            "hadm_id",
            "charttime",
            "itemid",
            "valuenum"
        ],
        parse_dates=["charttime"],
        chunksize=CHUNK
    )
):

    chunk = chunk[
        chunk["hadm_id"].isin(hadm_id_set) &
        chunk["itemid"].isin(all_lab_ids)
    ].dropna(subset=["valuenum"])

    if len(chunk):
        lab_chunks.append(chunk)

    if i % 10 == 0:
        n = sum(len(x) for x in lab_chunks)
        print(
            f"   chunk {i:3d} | "
            f"relevant rows {n:,}"
        )

labs = pd.concat(
    lab_chunks,
    ignore_index=True
)

hadm2stay = dict(
    zip(
        cohort["hadm_id"],
        cohort["stay_id"]
    )
)

labs["stay_id"] = (
    labs["hadm_id"].map(hadm2stay)
)

labs = labs.dropna(
    subset=["stay_id"]
)

labs["stay_id"] = (
    labs["stay_id"].astype(int)
)

itemid2lab = {
    iid: feature
    for feature, ids in LAB_ITEMIDS.items()
    for iid in ids
}

labs["feature"] = (
    labs["itemid"].map(itemid2lab)
)

print(
    "   Final lab rows:",
    f"{len(labs):,}"
)

del lab_chunks
gc.collect()

# ============================================================
# 5. BUILD EXACT 48-HOUR MATRIX
# ============================================================

print("\n[5/7] Building exact original 48-hour matrix...")

N_PAT = len(cohort)

feat2idx = {
    f: i
    for i, f in enumerate(
        ALL_TEMPORAL_FEATURES
    )
}

stay2idx = {
    sid: i
    for i, sid in enumerate(
        cohort["stay_id"]
    )
}

stay2in = dict(
    zip(
        cohort["stay_id"],
        cohort["intime"]
    )
)

X_temporal_id = np.full(
    (
        N_PAT,
        SEQ_LEN,
        N_FEATURES
    ),
    np.nan,
    dtype=np.float32
)

events = pd.concat(
    [
        chart[
            [
                "stay_id",
                "charttime",
                "feature",
                "valuenum"
            ]
        ],
        labs[
            [
                "stay_id",
                "charttime",
                "feature",
                "valuenum"
            ]
        ]
    ],
    ignore_index=True
)

events["intime"] = (
    events["stay_id"].map(stay2in)
)

events = events.dropna(
    subset=["intime"]
)

events["hours"] = (
    (
        events["charttime"]
        - events["intime"]
    )
    .dt.total_seconds()
    / 3600
)

events = events[
    (events["hours"] >= 0) &
    (events["hours"] < SEQ_LEN)
].copy()

events["hbin"] = (
    events["hours"]
    .astype(int)
    .clip(0, SEQ_LEN - 1)
)

print(
    "   Events inside 0–48h:",
    f"{len(events):,}"
)

# Exact original aggregation
grp = (
    events
    .groupby(
        [
            "stay_id",
            "hbin",
            "feature"
        ]
    )["valuenum"]
    .median()
)

for (sid, hbin, feat), val in grp.items():

    pi = stay2idx.get(sid)
    fi = feat2idx.get(feat)

    if pi is not None and fi is not None:
        X_temporal_id[
            pi,
            int(hbin),
            fi
        ] = val

del chart, labs, events, grp
gc.collect()

# ============================================================
# 6. EXACT ORIGINAL MINIMUM-DATA FILTER
# ============================================================

print("\n[6/7] Applying EXACT original minimum-data filter...")

n_vital = (
    ~np.isnan(
        X_temporal_id[
            :,
            :,
            :len(VITAL_SIGNS)
        ]
    )
).sum(axis=(1, 2))

n_lab = (
    ~np.isnan(
        X_temporal_id[
            :,
            :,
            len(VITAL_SIGNS):
        ]
    )
).sum(axis=(1, 2))

keep = (
    (n_vital >= 6) &
    (n_lab >= 2)
)

final_ids = cohort.loc[
    keep,
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "real_admit_year",
        "label"
    ]
].copy()

final_ids["n_vital"] = n_vital[keep]
final_ids["n_lab"] = n_lab[keep]

final_ids = final_ids.reset_index(drop=True)

print(
    "\n   FINAL recovered N:",
    f"{len(final_ids):,}"
)

print(
    "   Final positives:",
    f"{int(final_ids['label'].sum()):,}"
)

print(
    "   Final prevalence:",
    f"{100*final_ids['label'].mean():.2f}%"
)

# ============================================================
# 7. VALIDATE AGAINST SAVED CACHE
# ============================================================

print("\n[7/7] Validating against saved processed cache...")

cache = np.load(
    CACHE_FILE,
    allow_pickle=True
)

cache_y = (
    np.asarray(cache["labels"])
    .reshape(-1)
    .astype(int)
)

cache_year = (
    np.asarray(cache["years"])
    .reshape(-1)
    .astype(int)
)

id_y = (
    final_ids["label"]
    .to_numpy()
    .astype(int)
)

id_year = (
    final_ids["real_admit_year"]
    .to_numpy()
    .astype(int)
)

n_match = (
    len(final_ids) ==
    len(cache_y)
)

label_sequence_match = (
    n_match and
    np.array_equal(
        id_y,
        cache_y
    )
)

year_sequence_match = (
    n_match and
    np.array_equal(
        id_year,
        cache_year
    )
)

print("\n" + "=" * 100)
print("EXACT CACHE VALIDATION")
print("=" * 100)

print(
    f"Recovered N             : "
    f"{len(final_ids):,}"
)

print(
    f"Cache N                 : "
    f"{len(cache_y):,}"
)

print(
    "N match                :",
    "✅ YES" if n_match else "❌ NO"
)

print(
    "Label sequence match   :",
    "✅ YES"
    if label_sequence_match
    else "❌ NO"
)

print(
    "Year sequence match    :",
    "✅ YES"
    if year_sequence_match
    else "❌ NO"
)

# ============================================================
# HARD STOP IF NOT EXACT
# ============================================================

if not (
    n_match and
    label_sequence_match and
    year_sequence_match
):

    print("\n" + "=" * 100)
    print("🛑 VALIDATION FAILED")
    print("=" * 100)

    print(
        "Do NOT report subject-overlap values."
    )

    print(
        "The reconstructed identifiers do not "
        "exactly reproduce the saved cache."
    )

else:

    print("\n✅ EXACT CACHE RECONSTRUCTION CONFIRMED.")

    # ========================================================
    # SPLIT ASSIGNMENT
    # ========================================================

    final_ids["Split"] = np.select(
        [
            final_ids[
                "real_admit_year"
            ].between(2008, 2015),

            final_ids[
                "real_admit_year"
            ].between(2016, 2017),

            final_ids[
                "real_admit_year"
            ].between(2018, 2019),
        ],
        [
            "Training",
            "Validation",
            "Test"
        ],
        default="Unknown"
    )

    # ========================================================
    # SPLIT SUMMARY
    # ========================================================

    summary_rows = []

    for split in [
        "Training",
        "Validation",
        "Test"
    ]:

        d = final_ids[
            final_ids["Split"] == split
        ]

        summary_rows.append({
            "Split": split,
            "Records": len(d),
            "Unique Subjects":
                d["subject_id"].nunique(),
            "Mortality Positive":
                int(d["label"].sum()),
            "Prevalence (%)":
                100 * d["label"].mean(),
            "Year Range":
                (
                    f"{int(d['real_admit_year'].min())}"
                    f"–"
                    f"{int(d['real_admit_year'].max())}"
                )
        })

    split_summary = pd.DataFrame(
        summary_rows
    )

    # ========================================================
    # SUBJECT SETS
    # ========================================================

    tr = set(
        final_ids.loc[
            final_ids["Split"] ==
            "Training",
            "subject_id"
        ].tolist()
    )

    va = set(
        final_ids.loc[
            final_ids["Split"] ==
            "Validation",
            "subject_id"
        ].tolist()
    )

    te = set(
        final_ids.loc[
            final_ids["Split"] ==
            "Test",
            "subject_id"
        ].tolist()
    )

    overlap_tv = tr & va
    overlap_tt = tr & te
    overlap_vt = va & te

    overlap_table = pd.DataFrame([
        {
            "Comparison":
                "Training–Validation",
            "Subject Overlap":
                len(overlap_tv),
            "Status":
                "PASS"
                if len(overlap_tv) == 0
                else "OVERLAP DETECTED"
        },
        {
            "Comparison":
                "Training–Test",
            "Subject Overlap":
                len(overlap_tt),
            "Status":
                "PASS"
                if len(overlap_tt) == 0
                else "OVERLAP DETECTED"
        },
        {
            "Comparison":
                "Validation–Test",
            "Subject Overlap":
                len(overlap_vt),
            "Status":
                "PASS"
                if len(overlap_vt) == 0
                else "OVERLAP DETECTED"
        }
    ])

    # ========================================================
    # FULL REV-6 TABLE
    # ========================================================

    s = (
        split_summary
        .set_index("Split")
    )

    rev6_table = pd.DataFrame([
        {
            "Audit Item":
                "Cohort records",
            "Training":
                int(s.loc["Training", "Records"]),
            "Validation":
                int(s.loc["Validation", "Records"]),
            "Test":
                int(s.loc["Test", "Records"]),
            "Train–Val Overlap": "—",
            "Train–Test Overlap": "—",
            "Val–Test Overlap": "—",
            "Status": "PASS"
        },
        {
            "Audit Item":
                "Unique subjects",
            "Training":
                int(s.loc["Training", "Unique Subjects"]),
            "Validation":
                int(s.loc["Validation", "Unique Subjects"]),
            "Test":
                int(s.loc["Test", "Unique Subjects"]),
            "Train–Val Overlap":
                len(overlap_tv),
            "Train–Test Overlap":
                len(overlap_tt),
            "Val–Test Overlap":
                len(overlap_vt),
            "Status":
                (
                    "PASS"
                    if (
                        len(overlap_tv) == 0 and
                        len(overlap_tt) == 0 and
                        len(overlap_vt) == 0
                    )
                    else "OVERLAP DETECTED"
                )
        },
        {
            "Audit Item":
                "Admission years",
            "Training":
                s.loc["Training", "Year Range"],
            "Validation":
                s.loc["Validation", "Year Range"],
            "Test":
                s.loc["Test", "Year Range"],
            "Train–Val Overlap": "—",
            "Train–Test Overlap": "—",
            "Val–Test Overlap": "—",
            "Status": "PASS"
        },
        {
            "Audit Item":
                "Mortality positive, n",
            "Training":
                int(s.loc["Training", "Mortality Positive"]),
            "Validation":
                int(s.loc["Validation", "Mortality Positive"]),
            "Test":
                int(s.loc["Test", "Mortality Positive"]),
            "Train–Val Overlap": "—",
            "Train–Test Overlap": "—",
            "Val–Test Overlap": "—",
            "Status": "PASS"
        },
        {
            "Audit Item":
                "Mortality prevalence",
            "Training":
                f'{s.loc["Training", "Prevalence (%)"]:.1f}%',
            "Validation":
                f'{s.loc["Validation", "Prevalence (%)"]:.1f}%',
            "Test":
                f'{s.loc["Test", "Prevalence (%)"]:.1f}%',
            "Train–Val Overlap": "—",
            "Train–Test Overlap": "—",
            "Val–Test Overlap": "—",
            "Status": "PASS"
        },
        {
            "Audit Item":
                "Subject-ID overlap",
            "Training": "—",
            "Validation": "—",
            "Test": "—",
            "Train–Val Overlap":
                len(overlap_tv),
            "Train–Test Overlap":
                len(overlap_tt),
            "Val–Test Overlap":
                len(overlap_vt),
            "Status":
                (
                    "PASS"
                    if (
                        len(overlap_tv) == 0 and
                        len(overlap_tt) == 0 and
                        len(overlap_vt) == 0
                    )
                    else "OVERLAP DETECTED"
                )
        }
    ])

    print("\n" + "=" * 100)
    print("REV-6 — SPLIT CHARACTERISTICS")
    print("=" * 100)

    display(split_summary)

    print("\n" + "=" * 100)
    print("REV-6 — SUBJECT-LEVEL OVERLAP")
    print("=" * 100)

    display(overlap_table)

    print("\n" + "=" * 100)
    print("REV-6 — REPRODUCIBILITY AND DATA-LEAKAGE AUDIT")
    print("=" * 100)

    display(rev6_table)

    # ========================================================
    # SAVE EVERYTHING
    # ========================================================

    final_ids.to_csv(
        OUT_DIR /
        "REV6_FINAL_exact_identifiers.csv",
        index=False
    )

    split_summary.to_csv(
        OUT_DIR /
        "REV6_FINAL_split_summary.csv",
        index=False
    )

    overlap_table.to_csv(
        OUT_DIR /
        "REV6_FINAL_subject_overlap.csv",
        index=False
    )

    rev6_table.to_csv(
        OUT_DIR /
        "REV6_FINAL_audit_table.csv",
        index=False
    )

    # Save actual overlapping IDs if any
    pd.DataFrame({
        "subject_id":
            sorted(overlap_tv)
    }).to_csv(
        OUT_DIR /
        "REV6_overlap_train_validation.csv",
        index=False
    )

    pd.DataFrame({
        "subject_id":
            sorted(overlap_tt)
    }).to_csv(
        OUT_DIR /
        "REV6_overlap_train_test.csv",
        index=False
    )

    pd.DataFrame({
        "subject_id":
            sorted(overlap_vt)
    }).to_csv(
        OUT_DIR /
        "REV6_overlap_validation_test.csv",
        index=False
    )

    print("\n✅ ALL REV-6 RESULTS SAVED TO:")
    print(OUT_DIR)

    print("\n" + "=" * 100)

    if (
        len(overlap_tv) == 0 and
        len(overlap_tt) == 0 and
        len(overlap_vt) == 0
    ):
        print(
            "✅ FINAL AUDIT RESULT: "
            "NO CROSS-SPLIT SUBJECT OVERLAP."
        )
    else:
        print(
            "⚠️ FINAL AUDIT RESULT: "
            "CROSS-SPLIT SUBJECT OVERLAP DETECTED."
        )

    print("=" * 100)

print(
    f"\nElapsed: "
    f"{(time.time()-t_start)/60:.1f} minutes"
)

print("\n🛑 NO TRAINING WAS PERFORMED.")

In [ ]:
# CELL 3 — EXACT REVISION-I FEATURE DEFINITIONS
HF_ICD10_CODES = [f'I50{x}' for x in ['', '0', '1', '20', '21', '22', '23',
                                         '30', '31', '32', '33', '40', '41',
                                         '42', '43', '810', '811', '812',
                                         '813', '814', '89', '9']]
HF_ICD9_CODES  = ['428', '4280', '4281', '4289']

VITAL_SIGNS = ['heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'spo2', 'temperature']
LAB_VALUES = [
    'creatinine', 'bnp', 'lactate', 'troponin',
    'sodium', 'potassium', 'chloride', 'bicarbonate',
    'bun', 'glucose', 'hemoglobin', 'wbc',
    'platelets', 'inr', 'pt', 'ptt',
    'ph', 'pao2', 'paco2', 'fio2',
    'albumin', 'bilirubin_total', 'alt', 'ast'
]
STATIC_FEATURES = [
    'age', 'gender_male', 'admission_emergency',
    'admission_urgent', 'elixhauser_score',
    'hypertension', 'diabetes', 'ckd',
    'copd', 'af', 'prior_hf'
]
ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_VALUES
N_FEATURES = len(ALL_TEMPORAL_FEATURES)
N_STATIC   = len(STATIC_FEATURES)


print('N_FEATURES =', N_FEATURES, '| N_STATIC =', N_STATIC)


In [ ]:
# CELL 4 — EXACT COST IMPROVED CONFIGURATION FROM REVISION I
N_INPUT = N_FEATURES * 3   # value + missingness mask + delta-t

CONFIG_V2 = {
    'hidden_size'    : 256,
    'n_lstm_layers'  : 2,
    'n_heads'        : 4,
    'attention_dim'  : 128,
    'salience_hidden': 128,
    'dropout'        : 0.3,
    'sparsity_tau'   : 0.5,
    'lambda_sparse'  : 5e-4,
    'lr'             : 3e-4,
    'weight_decay'   : 1e-4,
    'batch_size'     : 64,
    'max_epochs'     : 120,
    'patience'       : 15,
    'grad_clip'      : 1.0,
    'warmup_pct'     : 0.1,
    'focal_gamma'    : 2.0,
    'focal_alpha'    : 0.75,
    'label_smoothing': 0.05,
    'stoch_depth_p'  : 0.1,
    'rdrop_alpha'    : 4.0,
    'n_snapshots'    : 5,
    'n_runs'         : 3,
    'seq_len'        : 48,
    'train_years'    : range(2008, 2016),
    'val_years'      : range(2016, 2018),
    'test_years'     : range(2018, 2020),
}

print('\u2705 Konfigurasi v2 dimuat')
print(f'   Input dim per timestep : {N_INPUT} (nilai + mask + delta_t)')
print(f'   Hidden size            : {CONFIG_V2["hidden_size"]}')


In [ ]:
# CELL 5 — EXACT COST IMPROVED PREPROCESSING + LOADERS FROM REVISION I
# Reuse already-loaded raw arrays from Bagian 3 (no need to re-read cache from disk)
X_temporal_raw, X_static_raw, labels_raw, years_raw = X_temporal, X_static, labels, years

def add_missingness_features(X_temporal):
    n, seq, feats = X_temporal.shape
    mask = (~np.isnan(X_temporal)).astype(np.float32)

    delta = np.zeros_like(X_temporal, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X_temporal[i, t, k]):
                    last_obs = t
                    delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)

    values = X_temporal.copy()
    n_vital = len(VITAL_SIGNS)
    for i in range(n):
        for k in range(n_vital):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)

    X_enhanced = np.concatenate([values, mask, delta], axis=-1)
    return X_enhanced.astype(np.float32)


def preprocess_v2(X_temporal, X_static, labels, years, config):
    train_mask = np.isin(years, list(config['train_years']))
    val_mask   = np.isin(years, list(config['val_years']))
    test_mask  = np.isin(years, list(config['test_years']))

    splits_raw = {}
    for name, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        splits_raw[name] = {
            'X_temp': X_temporal[mask].copy(),
            'X_stat': X_static[mask].copy(),
            'y'     : labels[mask].copy()
        }
        print(f'   {name:5s}: {mask.sum():,} pasien | pos: {labels[mask].sum()} ({labels[mask].mean()*100:.1f}%)')

    print('\n\u23f3 MICE imputation lab values...')
    n_vital = len(VITAL_SIGNS)
    imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)

    tr = splits_raw['train']['X_temp']
    n_tr, seq, n_feat = tr.shape
    lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
    lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)
    imp.fit(lab_train)

    for name in splits_raw:
        arr = splits_raw[name]['X_temp']
        n_sp = arr.shape[0]
        lab_d = arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital)
        lab_d = np.nan_to_num(lab_d, nan=0.0, posinf=0.0, neginf=0.0)
        arr[:, :, n_vital:] = imp.transform(lab_d).reshape(n_sp, seq, n_feat - n_vital)
        splits_raw[name]['X_temp'] = arr
    print('   \u2705 MICE selesai')

    print('\n Menambahkan missingness mask & delta-t features...')
    splits = {}
    for name in splits_raw:
        enhanced = add_missingness_features(splits_raw[name]['X_temp'])
        splits[name] = {
            'X_temp': enhanced,
            'X_stat': splits_raw[name]['X_stat'].copy(),
            'y'     : splits_raw[name]['y'].copy()
        }
    print(f'   Input shape baru: {splits["train"]["X_temp"].shape}')

    tr_flat = splits['train']['X_temp'][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
    feat_mean = tr_flat.mean(axis=0)
    feat_std  = tr_flat.std(axis=0) + 1e-8
    stat_mean = splits['train']['X_stat'].mean(axis=0)
    stat_std  = splits['train']['X_stat'].std(axis=0) + 1e-8

    for name in splits:
        arr = splits[name]['X_temp'].copy()
        arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
        splits[name]['X_temp'] = arr
        splits[name]['X_stat'] = (splits[name]['X_stat'] - stat_mean) / stat_std

    norm_stats = {'feat_mean': feat_mean, 'feat_std': feat_std,
                  'stat_mean': stat_mean, 'stat_std': stat_std}
    return splits, norm_stats


print('\u2699\ufe0f Preprocessing v2...')
splits, norm_stats = preprocess_v2(X_temporal_raw, X_static_raw, labels_raw, years_raw, CONFIG_V2)

for name, data in splits.items():
    print(f'   {name}: X_temp={data["X_temp"].shape} | X_stat={data["X_stat"].shape}')


class ICUDataset(Dataset):
    def __init__(self, X_temporal, X_static, y):
        self.X_temp = torch.FloatTensor(X_temporal)
        self.X_stat = torch.FloatTensor(X_static)
        self.y      = torch.FloatTensor(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X_temp[idx], self.X_stat[idx], self.y[idx]


def make_loaders_v2(splits, batch_size):
    loaders = {}
    for name, data in splits.items():
        ds = ICUDataset(data['X_temp'], data['X_stat'], data['y'])
        if name == 'train':
            y = data['y']
            w = 1.0 / np.bincount(y.astype(int))[y.astype(int)]
            sampler = torch.utils.data.WeightedRandomSampler(w, len(y), replacement=True)
            loaders[name] = DataLoader(ds, batch_size=batch_size, sampler=sampler,
                                       num_workers=2, pin_memory=True)
        else:
            loaders[name] = DataLoader(ds, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)
    return loaders


loaders = make_loaders_v2(splits, CONFIG_V2['batch_size'])
print('\u2705 DataLoaders v2 siap')
for name, dl in loaders.items():
    print(f'   {name}: {len(dl)} batches')


In [ ]:
# CELL 6 — EXACT SUPPORTING ARCHITECTURE BLOCKS FROM REVISION I
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=256, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class StochasticDepth(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x, residual):
        if not self.training or self.p == 0:
            return x + residual
        keep = torch.bernoulli(torch.tensor(1 - self.p)).item()
        return x + residual * keep


class ClinicalSalienceV2(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_features):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim * 2)
        self.norm  = nn.LayerNorm(hidden_dim)
        self.fc2   = nn.Linear(hidden_dim, n_features)
        self.tau   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x_t, h_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)
        h = self.fc1(combined)
        h, gate = h.chunk(2, dim=-1)
        h = h * torch.sigmoid(gate)
        h = self.norm(h)
        s_t = torch.sigmoid(self.fc2(h))
        m_hard = (s_t > self.tau).float()
        m_t = m_hard - s_t.detach() + s_t
        return m_t, s_t


class TemporalMultiHeadAttn(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(d_model)
        self.sdrop = StochasticDepth(p=0.1)

    def forward(self, x):
        out, weights = self.attn(x, x, x)
        x = self.sdrop(x, out)
        x = self.norm(x)
        return x, weights


class StaticTemporalFusion(nn.Module):
    def __init__(self, d_temporal, n_static, d_out, n_heads=4, dropout=0.1):
        super().__init__()
        self.static_proj = nn.Linear(n_static, d_temporal)
        self.cross_attn  = nn.MultiheadAttention(embed_dim=d_temporal, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_temporal)
        self.ffn  = nn.Sequential(
            nn.Linear(d_temporal * 2, d_temporal), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_temporal, d_out)
        )

    def forward(self, static_feat, hidden_states):
        q = self.static_proj(static_feat).unsqueeze(1)
        ctx, attn_w = self.cross_attn(q, hidden_states, hidden_states)
        ctx = self.norm(ctx.squeeze(1))
        pooled = hidden_states.mean(dim=1)
        z = self.ffn(torch.cat([ctx, pooled], dim=-1))
        return z, attn_w.squeeze(1)



print('Supporting architecture ready')


In [ ]:
# CELL 7 — COST IMPROVED WITH ONE REVISION-II SWITCH ONLY

class DSRNNv2_GatingIsolation(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden,
                 dropout=0.3, stoch_depth_p=0.1, lambda_sparse=5e-4,
                 use_controlled_gating=True):
        super().__init__()
        self.hidden_size   = hidden_size
        self.n_lstm_layers = n_lstm_layers
        self.lambda_sparse = lambda_sparse
        self.bidir_factor  = 2
        self.use_controlled_gating = bool(use_controlled_gating)

        self.input_proj = nn.Sequential(
            nn.Linear(n_input, hidden_size), nn.LayerNorm(hidden_size),
            nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size, dropout=dropout * 0.5)
        self.salience = ClinicalSalienceV2(
            hidden_size + hidden_size, salience_hidden, hidden_size
        )
        self.lstm = nn.LSTM(
            hidden_size, hidden_size, num_layers=n_lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_lstm_layers > 1 else 0
        )
        self.bidir_proj = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size), nn.GELU()
        )
        self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(
            hidden_size, n_static, attention_dim, n_heads, dropout
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(attention_dim // 2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, p in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(p)
                    elif 'bias' in name:
                        nn.init.zeros_(p)

    def forward(self, x_temporal, x_static, return_masks=False):
        batch, seq, _ = x_temporal.shape
        x_proj = self.input_proj(x_temporal)
        x_proj = self.pos_enc(x_proj)

        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)
        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        hard_masks = []

        for t in range(seq):
            x_t = x_proj[:, t, :]

            if self.use_controlled_gating:
                m_t, s_t = self.salience(x_t, h_prev)
                x_gated = m_t * x_t + (1 - m_t) * x_t * 0.1
                sparsity_loss = sparsity_loss + m_t.norm(p=1)
                # m_t forward value is the hard STE mask (0/1).
                hard_masks.append(m_t.detach().unsqueeze(1))
            else:
                # ONLY CHANGE: bypass controlled gating.
                x_gated = x_t

            gated_seq.append(x_gated.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_gated.detach() * 0.1

        x_gated_seq = torch.cat(gated_seq, dim=1)

        if self.use_controlled_gating:
            sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)
        else:
            sparsity_loss = torch.tensor(0.0, device=x_temporal.device)

        lstm_out, _ = self.lstm(x_gated_seq)
        lstm_out = self.bidir_proj(lstm_out)
        H, attn_weights = self.temporal_attn(lstm_out)
        z, _ = self.fusion(x_static, H)
        logits = self.classifier(z).squeeze(-1)

        if return_masks:
            masks = torch.cat(hard_masks, dim=1) if hard_masks else None
            return logits, sparsity_loss, masks
        return logits, sparsity_loss


def build_gating_model(config, gating):
    return DSRNNv2_GatingIsolation(
        n_input=N_INPUT, n_static=N_STATIC,
        hidden_size=config['hidden_size'],
        n_lstm_layers=config['n_lstm_layers'],
        n_heads=config['n_heads'],
        attention_dim=config['attention_dim'],
        salience_hidden=config['salience_hidden'],
        dropout=config['dropout'],
        stoch_depth_p=config['stoch_depth_p'],
        lambda_sparse=config['lambda_sparse'],
        use_controlled_gating=gating
    )

print("Gating-isolation model defined — NO TRAINING")


In [ ]:
# CELL 8 — EXACT REVISION-I LOSS/METRICS + MATCHED TRAINING

class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.smoothing = alpha, gamma, smoothing

    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        bce = F.binary_cross_entropy_with_logits(
            logits, targets_smooth, reduction='none'
        )
        prob = torch.sigmoid(logits)
        p_t = prob * targets + (1 - prob) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()

def rdrop_kl_loss(logits1, logits2):
    p1 = torch.sigmoid(logits1)
    p2 = torch.sigmoid(logits2)
    eps = 1e-8
    kl1 = p1 * (torch.log(p1 + eps) - torch.log(p2 + eps))
    kl2 = p2 * (torch.log(p2 + eps) - torch.log(p1 + eps))
    return ((kl1 + kl2) * 0.5).mean()

def sensitivity_at_specificity(y_true, y_prob, target_spec=0.90):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    idx = np.argmin(np.abs(spec - target_spec))
    return float(tpr[idx])

def compute_metrics(y_true, y_prob):
    return {
        'auc': roc_auc_score(y_true, y_prob),
        'auprc': average_precision_score(y_true, y_prob),
        'brier': brier_score_loss(y_true, y_prob),
        'sens_at_90spec': sensitivity_at_specificity(y_true, y_prob)
    }

@torch.no_grad()
def evaluate_r2(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x_temp, x_stat, y in loader:
        x_temp, x_stat = x_temp.to(DEVICE), x_stat.to(DEVICE)
        logits, _ = model(x_temp, x_stat)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

class SnapshotEnsembleR2:
    def __init__(self, k=5):
        self.k = k
        self.snapshots = []

    def update(self, val_auc, model):
        state = {k: v.detach().cpu().clone()
                 for k, v in model.state_dict().items()}
        self.snapshots.append((float(val_auc), state))
        self.snapshots.sort(key=lambda x: x[0], reverse=True)
        self.snapshots = self.snapshots[:self.k]

    @torch.no_grad()
    def predict(self, model, loader):
        probs_all, labels = [], None
        for auc, state in self.snapshots:
            model.load_state_dict(state)
            model.to(DEVICE)
            y, p = evaluate_r2(model, loader)
            probs_all.append(p)
            if labels is None:
                labels = y
        return labels, np.mean(probs_all, axis=0), probs_all

def reset_train_sampler_seed(seed):
    # WeightedRandomSampler uses global torch RNG unless a generator was supplied.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_gating_isolation(config, gating, seed, verbose=True):
    reset_train_sampler_seed(seed)

    model = build_gating_model(config, gating).to(DEVICE)
    optimizer = AdamW(
        model.parameters(), lr=config['lr'],
        weight_decay=config['weight_decay'], betas=(0.9, 0.999)
    )
    total_steps = config['max_epochs'] * len(loaders['train'])
    scheduler = OneCycleLR(
        optimizer, max_lr=config['lr'], total_steps=total_steps,
        pct_start=config['warmup_pct'], anneal_strategy='cos'
    )
    focal_loss = FocalLossWithSmoothing(
        alpha=config['focal_alpha'],
        gamma=config['focal_gamma'],
        smoothing=config['label_smoothing']
    )
    scaler = GradScaler()
    ensemble = SnapshotEnsembleR2(k=config['n_snapshots'])

    best_val_auc = -np.inf
    patience_count = 0
    history = []

    for epoch in range(1, config['max_epochs'] + 1):
        model.train()
        total_loss = 0.0

        for x_temp, x_stat, y in loaders['train']:
            x_temp = x_temp.to(DEVICE)
            x_stat = x_stat.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            with autocast():
                logits1, sp_loss1 = model(x_temp, x_stat)
                logits2, sp_loss2 = model(x_temp, x_stat)

                task_loss = (
                    focal_loss(logits1, y) + focal_loss(logits2, y)
                ) * 0.5
                rdrop_loss = rdrop_kl_loss(logits1, logits2)
                sp_loss = (sp_loss1 + sp_loss2) * 0.5

                loss = (
                    task_loss +
                    config['rdrop_alpha'] * rdrop_loss +
                    sp_loss
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        val_labels, val_probs = evaluate_r2(model, loaders['val'])
        val_auc = roc_auc_score(val_labels, val_probs)
        val_auprc = average_precision_score(val_labels, val_probs)
        ensemble.update(val_auc, model)

        history.append({
            'epoch': epoch,
            'loss': total_loss / len(loaders['train']),
            'val_auc': val_auc,
            'val_auprc': val_auprc
        })

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_count = 0
        else:
            patience_count += 1

        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(
                f"Epoch {epoch:3d} | "
                f"loss={history[-1]['loss']:.4f} | "
                f"val_AUC={val_auc:.4f} | "
                f"val_AUPRC={val_auprc:.4f} | "
                f"best={best_val_auc:.4f}"
            )

        if patience_count >= config['patience']:
            if verbose:
                print("Early stopping at epoch", epoch)
            break

    # Best single snapshot
    best_state = ensemble.snapshots[0][1]
    model.load_state_dict(best_state)
    model.to(DEVICE)
    test_labels, test_probs_single = evaluate_r2(model, loaders['test'])
    metrics_single = compute_metrics(test_labels, test_probs_single)

    # Snapshot ensemble, same principle as Revision I
    _, test_probs_ensemble, _ = ensemble.predict(model, loaders['test'])
    metrics_ensemble = compute_metrics(test_labels, test_probs_ensemble)

    tag = f"gating_{'ON' if gating else 'OFF'}_seed{seed}"
    torch.save(best_state, OUT_DIR / f"{tag}_best.pt")
    torch.save(ensemble.snapshots, OUT_DIR / f"{tag}_snapshots.pt")
    np.savez_compressed(
        OUT_DIR / f"{tag}_predictions.npz",
        test_labels=test_labels,
        probs_single=test_probs_single,
        probs_ensemble=test_probs_ensemble,
        seed=np.array(seed),
        controlled_gating=np.array(bool(gating))
    )
    pd.DataFrame(history).to_csv(
        OUT_DIR / f"{tag}_history.csv", index=False
    )

    return {
        'seed': seed,
        'gating': gating,
        'single': metrics_single,
        'ensemble': metrics_ensemble,
        'model': model,
        'history': history
    }

print("Training utilities ready — NO TRAINING")


In [ ]:
# CELL 9 — STRUCTURAL + FORWARD SANITY (NO TRAINING)

torch.manual_seed(SEED)
model_on = build_gating_model(CONFIG_V2, True).to(DEVICE)
torch.manual_seed(SEED)
model_off = build_gating_model(CONFIG_V2, False).to(DEVICE)

n_on = sum(p.numel() for p in model_on.parameters())
n_off = sum(p.numel() for p in model_off.parameters())

print("ON params :", f"{n_on:,}")
print("OFF params:", f"{n_off:,}")
print("Same parameter count:", n_on == n_off)
print("Same state keys:", list(model_on.state_dict()) == list(model_off.state_dict()))

# Ensure identical initial weights when seeded identically.
same_init = all(
    torch.equal(model_on.state_dict()[k], model_off.state_dict()[k])
    for k in model_on.state_dict()
)
print("Identical initial weights:", same_init)

batch = next(iter(loaders['train']))
xt, xs, y = batch
xt, xs = xt[:4].to(DEVICE), xs[:4].to(DEVICE)

model_on.eval()
model_off.eval()
with torch.no_grad():
    lo, spo, mo = model_on(xt, xs, return_masks=True)
    lf, spf, mf = model_off(xt, xs, return_masks=True)

print("ON logits:", tuple(lo.shape), "| OFF logits:", tuple(lf.shape))
print("ON sparsity penalty:", float(spo))
print("OFF sparsity penalty:", float(spf))
print("ON mask shape:", None if mo is None else tuple(mo.shape))
print("OFF mask:", mf)

assert n_on == n_off
assert same_init
assert float(spf) == 0.0
assert mf is None
print("✅ Isolation sanity PASSED — NO TRAINING")


In [ ]:
# CELL 10 — SAFETY SWITCHES
RUN_TRAINING = True
RUN_OFF_SEED42 = False
RUN_MATCHED_ON_SEED42 = False
RUN_REMAINING_SEEDS = True

print("RUN_TRAINING =", RUN_TRAINING)
print("First target = gating OFF, seed 42")
print("No training has started.")


In [ ]:
# CELL 11 — FIRST NEW RUN: GATING OFF / SEED 42 ONLY
if RUN_TRAINING and RUN_OFF_SEED42:
    t0 = time.time()
    result_off_42 = train_gating_isolation(
        CONFIG_V2, gating=False, seed=42, verbose=True
    )
    print("\nOFF seed42 single:", result_off_42['single'])
    print("OFF seed42 ensemble:", result_off_42['ensemble'])
    print(f"Elapsed: {(time.time()-t0)/60:.1f} minutes")
else:
    print("🔒 SKIPPED. Set RUN_TRAINING=True only after Cell 9 passes.")


In [ ]:
# CELL 12 — OPTIONAL MATCHED ON / SEED 42
# Run only after OFF seed42 is reviewed.
if RUN_TRAINING and RUN_MATCHED_ON_SEED42:
    t0 = time.time()
    result_on_42 = train_gating_isolation(
        CONFIG_V2, gating=True, seed=42, verbose=True
    )
    print("\nON seed42 single:", result_on_42['single'])
    print("ON seed42 ensemble:", result_on_42['ensemble'])
    print(f"Elapsed: {(time.time()-t0)/60:.1f} minutes")
else:
    print("🔒 Matched ON seed42 disabled.")


In [ ]:
# ENABLE ONLY SEEDS 43–44
RUN_TRAINING = True
RUN_REMAINING_SEEDS = True

# Pastikan seed 42 tidak dijalankan ulang
RUN_OFF_SEED42 = False
RUN_MATCHED_ON_SEED42 = False

print("RUN_TRAINING =", RUN_TRAINING)
print("RUN_REMAINING_SEEDS =", RUN_REMAINING_SEEDS)
print("RUN_OFF_SEED42 =", RUN_OFF_SEED42)
print("RUN_MATCHED_ON_SEED42 =", RUN_MATCHED_ON_SEED42)

In [ ]:
# CELL 13 — OPTIONAL SEEDS 43–44 FOR BOTH ARMS
# Enable only after seed42 results are accepted.
all_revision2_results = []

if RUN_TRAINING and RUN_REMAINING_SEEDS:
    for seed in [43, 44]:
        for gating in [True, False]:
            t0 = time.time()
            r = train_gating_isolation(
                CONFIG_V2, gating=gating, seed=seed, verbose=True
            )
            all_revision2_results.append({
                'seed': seed,
                'gating': 'ON' if gating else 'OFF',
                **r['ensemble']
            })
            print(
                f"{'ON' if gating else 'OFF'} seed {seed}: "
                f"{(time.time()-t0)/60:.1f} min"
            )

    display(pd.DataFrame(all_revision2_results))
else:
    print("🔒 Seeds 43–44 disabled.")


In [ ]:
# CELL 14 — ACTUAL SPARSITY EXTRACTION (NO RETRAINING)

@torch.no_grad()
def collect_hard_masks(model, loader):
    model.eval()
    all_masks = []
    for xt, xs, _ in loader:
        xt, xs = xt.to(DEVICE), xs.to(DEVICE)
        _, _, masks = model(xt, xs, return_masks=True)
        if masks is None:
            raise ValueError("Model has gating OFF; no masks available.")
        all_masks.append(masks.cpu().numpy())
    return np.concatenate(all_masks, axis=0)

def sparsity_summary(hard_masks):
    m = np.asarray(hard_masks, dtype=np.float32)
    retention = m.mean()
    patient_sparsity = 1.0 - m.mean(axis=(1,2))
    feature_sparsity = 1.0 - m.mean(axis=(0,1))
    timestep_sparsity = 1.0 - m.mean(axis=(0,2))
    return {
        'retention': float(retention),
        'sparsity': float(1-retention),
        'patient_mean': float(patient_sparsity.mean()),
        'patient_sd': float(patient_sparsity.std(ddof=1)),
        'q05_q25_q50_q75_q95': np.quantile(
            patient_sparsity, [0.05,0.25,0.5,0.75,0.95]
        ),
        'feature_sparsity': feature_sparsity,
        'timestep_sparsity': timestep_sparsity
    }

print("Sparsity extraction ready.")


### Run order
Run Cells **1–10** first. Cell 9 must print `Isolation sanity PASSED`.
Then change only `RUN_TRAINING = True` and run Cell 11 for **OFF seed 42**.
Do not run Cells 12–13 until the seed-42 result and runtime are reviewed.


In [ ]:
# ============================================================
# CHECK REVISION-II SAVED RESULTS
# ============================================================

from pathlib import Path
import os

RESULT_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/revision2_outputs"
)

print("RESULT DIRECTORY:")
print(RESULT_DIR)
print("\nFILES FOUND:")
print("=" * 70)

files = sorted(RESULT_DIR.glob("*"))

if not files:
    print("❌ No saved result files found.")
else:
    for f in files:
        size_mb = f.stat().st_size / (1024**2)
        print(f"{f.name:<50} {size_mb:>8.2f} MB")

print("=" * 70)
print("Total files:", len(files))

In [ ]:
# FIX — restore path for computational analysis
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/revision2_outputs"
)

print("RESULT_DIR :", RESULT_DIR)
print("Exists     :", RESULT_DIR.exists())

# Check required checkpoints
for f in [
    "gating_OFF_seed42_best.pt",
    "gating_ON_seed42_best.pt",
]:
    p = RESULT_DIR / f
    print(f"{f}: {'✅ FOUND' if p.exists() else '❌ MISSING'}")

In [ ]:
# ============================================================
# CELL 15 — TABLE XXV: ACTUAL CONTROLLED-SPARSITY
# NO TRAINING — checkpoint inference only
# ============================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/revision2_outputs"
)

SEEDS_R2 = [42, 43, 44]

@torch.no_grad()
def get_sparsity_from_checkpoint(seed):
    ckpt = RESULT_DIR / f"gating_ON_seed{seed}_best.pt"
    assert ckpt.exists(), f"Missing checkpoint: {ckpt}"

    model = build_gating_model(CONFIG_V2, True).to(DEVICE)

    state = torch.load(
        ckpt,
        map_location=DEVICE,
        weights_only=True
    )
    model.load_state_dict(state)
    model.eval()

    all_patient_sparsity = []
    retained = 0.0
    total = 0

    for x_temp, x_stat, _ in loaders["test"]:
        x_temp = x_temp.to(DEVICE)
        x_stat = x_stat.to(DEVICE)

        _, _, masks = model(
            x_temp,
            x_stat,
            return_masks=True
        )

        assert masks is not None

        m = masks.float()

        retained += m.sum().item()
        total += m.numel()

        patient_sp = 1.0 - m.mean(dim=(1, 2))
        all_patient_sparsity.append(
            patient_sp.cpu().numpy()
        )

    patient_sp = np.concatenate(all_patient_sparsity)

    retention = retained / total
    sparsity = 1.0 - retention

    return {
        "seed": seed,
        "retention_pct": retention * 100,
        "sparsity_pct": sparsity * 100,
        "patient_mean_pct": patient_sp.mean() * 100,
        "patient_sd_pct": patient_sp.std(ddof=1) * 100,
        "median_pct": np.median(patient_sp) * 100,
        "q25_pct": np.quantile(patient_sp, .25) * 100,
        "q75_pct": np.quantile(patient_sp, .75) * 100,
        "q05_pct": np.quantile(patient_sp, .05) * 100,
        "q95_pct": np.quantile(patient_sp, .95) * 100,
    }


sparsity_rows = []

for seed in SEEDS_R2:
    print(f"Analyzing Gating ON seed {seed}...")
    r = get_sparsity_from_checkpoint(seed)
    sparsity_rows.append(r)

sparsity_df = pd.DataFrame(sparsity_rows)

print("\nPER-SEED RESULTS")
display(sparsity_df.round(4))


# ============================================================
# Aggregate across the three independently trained models
# ============================================================

ret_mean = sparsity_df["retention_pct"].mean()
ret_sd   = sparsity_df["retention_pct"].std(ddof=1)

sp_mean = sparsity_df["sparsity_pct"].mean()
sp_sd   = sparsity_df["sparsity_pct"].std(ddof=1)

patient_mean = sparsity_df["patient_mean_pct"].mean()
patient_sd_across_models = sparsity_df["patient_mean_pct"].std(ddof=1)

median_mean = sparsity_df["median_pct"].mean()
q25_mean = sparsity_df["q25_pct"].mean()
q75_mean = sparsity_df["q75_pct"].mean()

table_xxv = pd.DataFrame([
    {
        "Configuration": "Gating OFF",
        "Retention (%)": "100.00",
        "Sparsity (%)": "0.00",
        "Patient Mean Sparsity (%)": "—",
        "Median (%)": "—",
        "IQR (%)": "—"
    },
    {
        "Configuration": "Gating ON",
        "Retention (%)":
            f"{ret_mean:.2f} ± {ret_sd:.2f}",
        "Sparsity (%)":
            f"{sp_mean:.2f} ± {sp_sd:.2f}",
        "Patient Mean Sparsity (%)":
            f"{patient_mean:.2f} ± {patient_sd_across_models:.2f}",
        "Median (%)":
            f"{median_mean:.2f}",
        "IQR (%)":
            f"{q25_mean:.2f}–{q75_mean:.2f}"
    }
])

print("\n" + "="*90)
print("TABLE XXV — ACTUAL CONTROLLED-SPARSITY CHARACTERISTICS")
print("="*90)

display(table_xxv)

sparsity_df.to_csv(
    RESULT_DIR / "table_xxv_sparsity_per_seed.csv",
    index=False
)

table_xxv.to_csv(
    RESULT_DIR / "table_xxv_sparsity_summary.csv",
    index=False
)

print("✅ TABLE XXV saved.")

In [ ]:
# ============================================================
# PREFLIGHT CELL 16 — CHECK ALL DEPENDENCIES
# NO TRAINING
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import time

RESULT_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/revision2_outputs"
)

checks = {
    "RESULT_DIR": RESULT_DIR.exists(),
    "build_gating_model": "build_gating_model" in globals(),
    "CONFIG_V2": "CONFIG_V2" in globals(),
    "DEVICE": "DEVICE" in globals(),
    "loaders": "loaders" in globals(),
    "sp_mean": "sp_mean" in globals(),
}

for k, v in checks.items():
    print(f"{k:20s}: {'✅' if v else '❌'}")

print("\nCHECKPOINTS")
for gating in ["ON", "OFF"]:
    p = RESULT_DIR / f"gating_{gating}_seed42_best.pt"
    print(f"{p.name:35s}: {'✅' if p.exists() else '❌'}")

# Restore sparsity automatically if variable disappeared
if "sp_mean" not in globals():
    sf = RESULT_DIR / "table_xxv_sparsity_per_seed.csv"
    if sf.exists():
        tmp = pd.read_csv(sf)
        sp_mean = tmp["sparsity_pct"].mean()
        print(f"\n✅ sp_mean restored from CSV: {sp_mean:.4f}%")
    else:
        print("\n❌ Sparsity CSV not found")

print("\nFINAL STATUS")
required = [
    "build_gating_model" in globals(),
    "CONFIG_V2" in globals(),
    "DEVICE" in globals(),
    "loaders" in globals(),
    "sp_mean" in globals(),
]

if all(required):
    print("✅ READY FOR TABLE XXVI — NO TRAINING REQUIRED")
else:
    print("🛑 NOT READY — see ❌ items above")

In [ ]:
# ============================================================
# CELL 16 — TABLE XXVI: COMPUTATIONAL CHARACTERISTICS
# NO TRAINING — BENCHMARK ONLY
# ============================================================

import time
import numpy as np
import pandas as pd
import torch


def model_parameter_stats(model):
    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    memory_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )

    return (
        total_params,
        trainable_params,
        memory_bytes / (1024 ** 2)
    )


@torch.no_grad()
def benchmark_latency(
    model,
    loader,
    warmup_batches=10,
    measured_batches=50
):
    model.eval()

    # Same representative batch for ON and OFF
    x_temp, x_stat, _ = next(iter(loader))
    x_temp = x_temp.to(DEVICE)
    x_stat = x_stat.to(DEVICE)

    batch_size = x_temp.shape[0]

    # GPU warm-up
    for _ in range(warmup_batches):
        _ = model(x_temp, x_stat)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    times = []

    for _ in range(measured_batches):

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        _ = model(x_temp, x_stat)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        # ms per patient
        times.append(
            elapsed * 1000.0 / batch_size
        )

    times = np.asarray(times)

    return {
        "latency_mean_ms": times.mean(),
        "latency_sd_ms": times.std(ddof=1),
        "latency_median_ms": np.median(times),
        "latency_p25_ms": np.quantile(times, 0.25),
        "latency_p75_ms": np.quantile(times, 0.75),
    }


computational_rows = []

for gating in [False, True]:

    label = (
        "Improved COST – ON"
        if gating
        else "Improved COST – OFF"
    )

    ckpt = RESULT_DIR / (
        "gating_ON_seed42_best.pt"
        if gating
        else "gating_OFF_seed42_best.pt"
    )

    print(f"\nBenchmarking {label}...")
    print("Checkpoint:", ckpt)

    assert ckpt.exists(), f"Missing checkpoint: {ckpt}"

    model = build_gating_model(
        CONFIG_V2,
        gating
    ).to(DEVICE)

    state = torch.load(
        ckpt,
        map_location=DEVICE,
        weights_only=True
    )

    model.load_state_dict(state)
    model.eval()

    params, trainable, memory_mb = model_parameter_stats(model)

    latency = benchmark_latency(
        model,
        loaders["test"],
        warmup_batches=10,
        measured_batches=50
    )

    computational_rows.append({
        "Configuration": label,
        "Parameters": params,
        "Trainable Parameters": trainable,
        "Model Memory (MB)": memory_mb,
        **latency
    })

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


comp_df = pd.DataFrame(computational_rows)


# ============================================================
# Relative latency
# ============================================================

off_latency = comp_df.loc[
    comp_df["Configuration"] == "Improved COST – OFF",
    "latency_mean_ms"
].iloc[0]

comp_df["Relative Latency"] = (
    comp_df["latency_mean_ms"] / off_latency
)


# ============================================================
# Add actual sparsity from TABLE XXV
# ============================================================

comp_df["Sparsity (%)"] = [
    0.0,
    sp_mean
]


# ============================================================
# PAPER-READY TABLE
# ============================================================

table_xxvi = pd.DataFrame({
    "Configuration":
        comp_df["Configuration"],

    "Parameters":
        comp_df["Parameters"].map(
            lambda x: f"{x:,}"
        ),

    "Model Memory (MB)":
        comp_df["Model Memory (MB)"].map(
            lambda x: f"{x:.2f}"
        ),

    "Inference Latency (ms/patient)":
        comp_df.apply(
            lambda r:
            f'{r["latency_mean_ms"]:.3f} ± '
            f'{r["latency_sd_ms"]:.3f}',
            axis=1
        ),

    "Relative Latency":
        comp_df["Relative Latency"].map(
            lambda x: f"{x:.2f}×"
        ),

    "Sparsity (%)": [
        "0.00",
        f"{sp_mean:.2f}"
    ]
})


print("\n" + "=" * 105)
print("TABLE XXVI — COMPUTATIONAL CHARACTERISTICS OF CONTROLLED GATING")
print("=" * 105)

display(table_xxvi)


print("\nDETAILED LATENCY DISTRIBUTION")

display(
    comp_df[[
        "Configuration",
        "latency_mean_ms",
        "latency_sd_ms",
        "latency_median_ms",
        "latency_p25_ms",
        "latency_p75_ms"
    ]].round(4)
)


# ============================================================
# SAVE
# ============================================================

comp_df.to_csv(
    RESULT_DIR /
    "table_xxvi_computational_detailed.csv",
    index=False
)

table_xxvi.to_csv(
    RESULT_DIR /
    "table_xxvi_computational_summary.csv",
    index=False
)

print("\n✅ TABLE XXVI saved.")
print("🛑 NO TRAINING WAS PERFORMED.")

In [ ]:
# ============================================================
# SANITY CHECK — IS 99% SPARSITY REAL?
# NO TRAINING
# ============================================================

import torch
import numpy as np

seed = 42

model = build_gating_model(CONFIG_V2, True).to(DEVICE)

ckpt = RESULT_DIR / f"gating_ON_seed{seed}_best.pt"
state = torch.load(ckpt, map_location=DEVICE, weights_only=True)
model.load_state_dict(state)
model.eval()

x_temp, x_stat, _ = next(iter(loaders["test"]))
x_temp = x_temp.to(DEVICE)
x_stat = x_stat.to(DEVICE)

with torch.no_grad():
    output = model(x_temp, x_stat, return_masks=True)

print("Number of outputs:", len(output))

logits, sp_loss, masks = output

m = masks.detach().float().cpu().numpy()

print("\n========== MASK SANITY CHECK ==========")
print("Shape              :", m.shape)
print("Min                :", m.min())
print("Max                :", m.max())
print("Mean               :", m.mean())
print("Std                :", m.std())

unique = np.unique(m)
print("Number unique vals :", len(unique))
print("First unique vals  :", unique[:20])

print("\n--- EXACT VALUES ---")
print("% exactly zero     :", (m == 0).mean() * 100)
print("% exactly one      :", (m == 1).mean() * 100)

print("\n--- THRESHOLD CHECK ---")
for threshold in [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9]:
    retained = (m >= threshold).mean() * 100
    print(
        f"threshold >= {threshold:.2f}: "
        f"retained={retained:.3f}% | "
        f'sparsity={100-retained:.3f}%'
    )

print("\n--- SAMPLE MASK ---")
print(m[0, :5, :10])

print("\nSparsity loss:", float(sp_loss))

print("\n🛑 NO TRAINING PERFORMED")

In [ ]:
# REV-6A — FIND SUBJECT-ID VARIABLE
# NO TRAINING

candidates = []

for name, obj in globals().items():
    lname = name.lower()

    if any(k in lname for k in [
        "subject", "patient", "stay_id",
        "hadm", "ids", "id_"
    ]):
        try:
            shape = getattr(obj, "shape", None)
            length = len(obj) if hasattr(obj, "__len__") else None

            candidates.append({
                "variable": name,
                "type": type(obj).__name__,
                "shape": str(shape),
                "length": length
            })
        except:
            pass

import pandas as pd

df_candidates = pd.DataFrame(candidates)

print("POSSIBLE ID VARIABLES:")
display(df_candidates)

if len(df_candidates) == 0:
    print("❌ No subject/patient ID variable found in current runtime.")
else:
    print("✅ Possible ID variable(s) found.")

print("\nNO TRAINING PERFORMED.")